In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor
import joblib

# 1. LOAD DATA
df = pd.read_csv("../slippage_data.csv")

# 2. DATA CLEANING
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()
print(f"Data loaded after cleaning: {len(df)} trades.")

# 3. PREPARE THE DATA & FEATURE ENGINEERING
df['side_encoded'] = df['side'].map({'BUY': 1, 'SELL': 0})

# Existing feature: Consumption Ratio
df['depth_to_consume'] = df.apply(lambda row: row['book_ask_depth'] if row['side'] == 'BUY' else row['book_bid_depth'], axis=1)
df['consumption_ratio'] = df['requested_size'] / (df['depth_to_consume'] + 1e-9) 

# NEW FEATURE: Book Imbalance
# We add 1e-9 to the denominator to prevent division by zero if the book is empty
df['book_imbalance'] = df['book_bid_depth'] / (df['book_bid_depth'] + df['book_ask_depth'] + 1e-9)

# 4. SELECT FEATURES AND TARGET
# Notice we added 'book_imbalance' to the X list!
X = df[['requested_size', 'side_encoded', 'book_bid_depth', 'book_ask_depth', 'spread', 'consumption_ratio', 'book_imbalance']]
y = df['slippage']

# 5. SPLIT INTO TRAIN AND TEST
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 6. TRAIN THE XGBOOST MODEL (1,000 trees)
print("Training XGBoost with 1,000 estimators and Book Imbalance...")
model = XGBRegressor(
    n_estimators=1000, 
    learning_rate=0.05, 
    max_depth=6, 
    random_state=42,
    n_jobs=-1 
)

model.fit(X_train, y_train)

# 7. EVALUATE THE MODEL
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"\n--- MODEL EVALUATION ---")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R2 Score: {r2:.4f}")

# 8. FEATURE IMPORTANCE
print("\n--- FEATURE IMPORTANCE ---")
for feature, imp in zip(X.columns, model.feature_importances_):
    print(f"{feature}: {imp:.4f}")

# 9. SAVE THE NEW MODEL
joblib.dump(model, "../results/slippage_model.pkl")
print("\n XGBoost Model saved successfully with Book Imbalance feature!")

Data loaded after cleaning: 31463 trades.
Training XGBoost with 1,000 estimators and Book Imbalance...

--- MODEL EVALUATION ---
Mean Absolute Error (MAE): 0.0918
R2 Score: 0.3644

--- FEATURE IMPORTANCE ---
requested_size: 0.0898
side_encoded: 0.0644
book_bid_depth: 0.1005
book_ask_depth: 0.1021
spread: 0.1483
consumption_ratio: 0.4000
book_imbalance: 0.0949

 XGBoost Model saved successfully with Book Imbalance feature!
